# 01 - Data Exploration and Preliminary Preprocessing

Main thing to check here: what the OrionCAF Turkish legal QA dataset looks like, and how we can turn it into searchable passages for our RAG system.

This is mostly a working notebook. After we agree on the cleaning choices, the reusable parts can be moved into `src/rag_turkish_law/data/`.

## What We Are Checking

1. Load the dataset from Hugging Face.
2. Look at columns, examples, missing values, and repeated questions.
3. Convert each row into a consistent `question`, `answer`, `passage_text`, and metadata format.
4. Filter rows that are probably not useful for retrieval.
5. Save a first passage file and a small seed evaluation file.

In [15]:
# Install these if the notebook environment does not have them yet:
# %pip install datasets pandas numpy tqdm pyarrow

from __future__ import annotations

import json
import random
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import load_dataset

In [16]:
DATASET_NAME = "OrionCAF/turkish_law_qa_dataset"
SPLIT = "train"
SEED = 42

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
EVAL_DIR = ROOT / "data" / "evaluation"

for path in [RAW_DIR, PROCESSED_DIR, EVAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
ROOT

PosixPath('/Users/onur/Desktop/CS455/Term Project')

## Load Dataset

This may download the dataset the first time it runs. For the report, we should record the dataset name, split, access date, license, and filtering choices.

In [17]:
dataset = load_dataset(DATASET_NAME, split=SPLIT)
dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 18305
})

In [18]:
print("rows:", len(dataset))
print("columns:", dataset.column_names)

example = dataset[0]
for key, value in example.items():
    preview = str(value).replace("\n", " ")[:300]
    print(f"\n[{key}]\n{preview}")

rows: 18305
columns: ['question', 'answer']

[question]
İstihkak davasının amacı nedir ve hangi durumlarda uygulanır?

[answer]
İstihkak davasının amacı, haksız olarak elde bulundurulan malın geri alınmasıdır ve bu durum taşınır mallar için geçerlidir.


## Convert To DataFrame For Quick Inspection

In [19]:
df = dataset.to_pandas()
df.head(3)

,question,answer
0,İstihkak davasının amacı nedir ve hangi duruml...,"İstihkak davasının amacı, haksız olarak elde b..."
1,Tapuya kayıtlı taşınmazlarda istihkak davasını...,Tapuya kayıtlı taşınmazlarda istihkak davasını...
2,"Gerçek malik tapu sicilinde görünüyorsa, hangi...","Gerçek malik tapu sicilinde görünüyorsa, istih..."


In [20]:
summary_rows = []
for col in df.columns:
    non_null = df[col].notna().sum()
    as_text = df[col].fillna("").astype(str)
    summary_rows.append(
        {
            "column": col,
            "non_null": int(non_null),
            "empty_strings": int((as_text.str.strip() == "").sum()),
            "mean_chars": round(float(as_text.str.len().mean()), 1),
            "max_chars": int(as_text.str.len().max()),
        }
    )

pd.DataFrame(summary_rows).sort_values("column")

,column,non_null,empty_strings,mean_chars,max_chars
1,answer,18305,0,155.4,660
0,question,18305,0,86.5,218


## Normalization Helpers

The dataset is simple right now, but these helpers keep the notebook flexible if field names change or if we add another dataset later.

In [21]:
QUESTION_FIELDS = ["question", "soru", "title", "baslik", "prompt"]
ANSWER_FIELDS = ["answer", "cevap", "response", "explanation", "aciklama", "content"]
CATEGORY_FIELDS = ["category", "kategori", "topic", "konu", "law_area"]


def clean_text(value) -> str:
    if value is None:
        return ""
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def first_non_empty(row: dict, fields: list[str]) -> tuple[str, str | None]:
    lowered = {k.lower(): k for k in row.keys()}
    for field in fields:
        real_key = lowered.get(field.lower())
        if real_key is None:
            continue
        value = clean_text(row.get(real_key))
        if value:
            return value, real_key
    return "", None


def normalize_record(row: dict, row_index: int) -> dict:
    question, question_field = first_non_empty(row, QUESTION_FIELDS)
    answer, answer_field = first_non_empty(row, ANSWER_FIELDS)
    category, category_field = first_non_empty(row, CATEGORY_FIELDS)

    if question and answer:
        passage_text = f"Soru: {question}\nCevap: {answer}"
    else:
        passage_text = answer or question

    return {
        "passage_id": f"orioncaf_{row_index}",
        "question": question,
        "answer": answer,
        "category": category,
        "passage_text": passage_text,
        "source_dataset": DATASET_NAME,
        "source_split": SPLIT,
        "source_row": row_index,
        "detected_fields": {
            "question": question_field,
            "answer": answer_field,
            "category": category_field,
        },
    }

In [22]:
records = [normalize_record(dict(row), i) for i, row in enumerate(dataset)]
records_df = pd.DataFrame(records)
records_df.head(3)

,passage_id,question,answer,category,passage_text,source_dataset,source_split,source_row,detected_fields
0,orioncaf_0,İstihkak davasının amacı nedir ve hangi duruml...,"İstihkak davasının amacı, haksız olarak elde b...",,Soru: İstihkak davasının amacı nedir ve hangi ...,OrionCAF/turkish_law_qa_dataset,train,0,"{'question': 'question', 'answer': 'answer', '..."
1,orioncaf_1,Tapuya kayıtlı taşınmazlarda istihkak davasını...,Tapuya kayıtlı taşınmazlarda istihkak davasını...,,Soru: Tapuya kayıtlı taşınmazlarda istihkak da...,OrionCAF/turkish_law_qa_dataset,train,1,"{'question': 'question', 'answer': 'answer', '..."
2,orioncaf_2,"Gerçek malik tapu sicilinde görünüyorsa, hangi...","Gerçek malik tapu sicilinde görünüyorsa, istih...",,"Soru: Gerçek malik tapu sicilinde görünüyorsa,...",OrionCAF/turkish_law_qa_dataset,train,2,"{'question': 'question', 'answer': 'answer', '..."


## Basic Cleaning Checks

In [23]:
records_df["question_chars"] = records_df["question"].str.len()
records_df["answer_chars"] = records_df["answer"].str.len()
records_df["passage_chars"] = records_df["passage_text"].str.len()

stats = {
    "total_records": len(records_df),
    "missing_question": int((records_df["question"].str.len() == 0).sum()),
    "missing_answer": int((records_df["answer"].str.len() == 0).sum()),
    "short_answers_lt_60_chars": int((records_df["answer_chars"] < 60).sum()),
    "duplicate_questions": int(records_df["question"].duplicated().sum()),
}

stats

{'total_records': 18305,
 'missing_question': 0,
 'missing_answer': 0,
 'short_answers_lt_60_chars': 120,
 'duplicate_questions': 270}

In [24]:
records_df[["question_chars", "answer_chars", "passage_chars"]].describe().round(1)

,question_chars,answer_chars,passage_chars
count,18305.0,18305.0,18305.0
mean,86.5,155.4,255.9
std,23.0,56.1,65.4
min,22.0,30.0,85.0
25%,70.0,116.0,210.0
50%,85.0,147.0,249.0
75%,101.0,185.0,293.0
max,218.0,660.0,754.0


In [25]:
usable_df = records_df[
    (records_df["question"].str.len() > 0)
    & (records_df["answer"].str.len() >= 60)
].copy()

usable_df = usable_df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

print("usable records:", len(usable_df))
usable_df[["passage_id", "question", "answer", "category"]].head(5)

usable records: 18066


,passage_id,question,answer,category
0,orioncaf_0,İstihkak davasının amacı nedir ve hangi duruml...,"İstihkak davasının amacı, haksız olarak elde b...",
1,orioncaf_1,Tapuya kayıtlı taşınmazlarda istihkak davasını...,Tapuya kayıtlı taşınmazlarda istihkak davasını...,
2,orioncaf_2,"Gerçek malik tapu sicilinde görünüyorsa, hangi...","Gerçek malik tapu sicilinde görünüyorsa, istih...",
3,orioncaf_3,İstihkak davasının esas fonksiyonu hangi tür m...,"İstihkak davasının esas fonksiyonu, taşınır ma...",
4,orioncaf_4,İstihkak davası nedir ve hangi durumlarda açılır?,"İstihkak davası, malikin mülkiyet hakkına daya...",


## Create Searchable Passage Objects

These passage objects are the format we can later embed and put into FAISS or Chroma.

In [26]:
def make_passage(row: pd.Series) -> dict:
    return {
        "passage_id": row["passage_id"],
        "text": row["passage_text"],
        "title": row["question"][:160],
        "snippet": row["answer"][:500],
        "tag": row["category"] or "legal_qa",
        "source_dataset": row["source_dataset"],
        "source_split": row["source_split"],
        "source_row": int(row["source_row"]),
    }


passages = [make_passage(row) for _, row in usable_df.iterrows()]
passages[:2]

[{'passage_id': 'orioncaf_0',
  'text': 'Soru: İstihkak davasının amacı nedir ve hangi durumlarda uygulanır?\nCevap: İstihkak davasının amacı, haksız olarak elde bulundurulan malın geri alınmasıdır ve bu durum taşınır mallar için geçerlidir.',
  'title': 'İstihkak davasının amacı nedir ve hangi durumlarda uygulanır?',
  'snippet': 'İstihkak davasının amacı, haksız olarak elde bulundurulan malın geri alınmasıdır ve bu durum taşınır mallar için geçerlidir.',
  'tag': 'legal_qa',
  'source_dataset': 'OrionCAF/turkish_law_qa_dataset',
  'source_split': 'train',
  'source_row': 0},
 {'passage_id': 'orioncaf_1',
  'text': 'Soru: Tapuya kayıtlı taşınmazlarda istihkak davasının işlevi nasıl değişir?\nCevap: Tapuya kayıtlı taşınmazlarda istihkak davasının işlevi, tapu sicilinde asıl malik kayıtlı değilse tapu sicilinin düzeltilmesi davası ile yerine getirilir.',
  'title': 'Tapuya kayıtlı taşınmazlarda istihkak davasının işlevi nasıl değişir?',
  'snippet': 'Tapuya kayıtlı taşınmazlarda istihka

## Create Initial Held-Out Seed

This is only a starter evaluation file so we can test retrieval. The real evaluation set still needs manually written ambiguous, unsupported, and legal-advice-risk questions.

In [27]:
heldout_size = min(50, len(passages))
heldout_indices = random.sample(range(len(passages)), heldout_size)

# Keep all passages in the searchable corpus. The held-out file stores
# evaluation questions and gold IDs, not documents to remove from retrieval.
corpus = passages
heldout = []
for idx in heldout_indices:
    passage = passages[idx]
    heldout.append(
        {
            "question": passage["title"],
            "gold_passage_id": passage["passage_id"],
            "kind": "dataset_title_seed",
            "notes": "Auto-created seed item; replace or augment with manual annotations.",
        }
    )

len(corpus), len(heldout)

(18066, 50)

## Write Preliminary Outputs

These generated files are ignored by git, so they stay local unless we later decide to commit a tiny sample.

In [28]:
def write_jsonl(rows: list[dict], path: Path) -> None:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


write_jsonl(corpus, PROCESSED_DIR / "passages_seed.jsonl")
write_jsonl(heldout, EVAL_DIR / "heldout_seed.jsonl")

report = {
    "dataset_name": DATASET_NAME,
    "split": SPLIT,
    "stats": stats,
    "usable_records": len(usable_df),
    "corpus_seed_size": len(corpus),
    "heldout_seed_size": len(heldout),
    "columns": list(df.columns),
}

(PROCESSED_DIR / "preprocessing_seed_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

report

{'dataset_name': 'OrionCAF/turkish_law_qa_dataset',
 'split': 'train',
 'stats': {'total_records': 18305,
  'missing_question': 0,
  'missing_answer': 0,
  'short_answers_lt_60_chars': 120,
  'duplicate_questions': 270},
 'usable_records': 18066,
 'corpus_seed_size': 18066,
 'heldout_seed_size': 50,
 'columns': ['question', 'answer']}

## Manual Annotation To Do

For the final evaluation, we still need a manually reviewed set with columns like:

- `question`
- `gold_passage_ids`
- `expected_support`: supported / partial / unsupported / insufficient / risk
- `legal_advice_risk`: yes / no
- `annotation_notes`

The proposal feedback mentioned manual annotation, so we should turn this into a short rubric in `evaluation/annotations/` or the final report notes.